# C4 · 캡스톤 4: 옵션 가격 결정 (Capstone 4: Monte Carlo Option Pricing with Variance Reduction)

| 항목 | 내용 |
|---|---|
| 교재 지도 | Shreve II §4.5 (블랙–숄즈–머튼), §7.3 (녹아웃 장벽 옵션), §7.5 (아시안 옵션) · Glasserman, *Monte Carlo Methods in Financial Engineering* (2004) §4.1 (대조 변량), §4.2 (반정 변량), §6.4 (장벽: 연속성 보정·브라운 다리), §7.2 (경로별 미분), §7.3 (우도비) · Øksendal Ch. 12 (블랙–숄즈) · Durrett 3e Ch. 6 (위험중립 가격) · Broadie–Glasserman–Kou (1997), *Math. Finance* 7, Thm 1.1 ($\beta = 0.5826$) |
| 선수 노트북 (재사용) | 07e (BS 공식·위험중립·`black_scholes_call`·`mc_option_price`), 07c (오일러–마루야마 vs 정확 GBM, 약수렴 편향), 00b (SE·log-log 오차), 06c (반사 원리·브라운 다리), 06e (`gbm_paths`), 07d (OU: Recall만) |
| 예상 소요 | 150분 |
| 상태 | draft |

## 프로젝트 브리프 — 합격 기준과 단계

**브리프.** 블랙–숄즈 세계($S_0 = 100$, $K = 100$, $r = 0.05$, $\sigma = 0.2$, $T = 1$, 위험중립 GBM $dS = rS\,dt + \sigma S\,dW$)에서 세 가지 옵션을 몬테카를로(MC)로 가격 결정한다: 유러피언 콜, 이산 관측 아시안 콜(산술·기하, $n = 50$ 관측일), 다운앤아웃 장벽 콜($H = 90$). 질문은 세 가지다. (i) 반정 변량(antithetic variates)과 대조 변량(control variates)이 분산을 **얼마나** 줄이는가(00b의 SE로 정량화). (ii) 장벽 옵션을 유한 격자로 관측할 때 생기는 이산화 편향은 얼마이고, BGK 연속성 보정과 브라운 다리 보정이 그것을 어떻게 지우는가(07c의 이산화 오차 관점). (iii) 그릭스(Δ, vega, Γ)를 경로별(pathwise)·우도비(likelihood ratio) 추정량으로 구하면 닫힌 형식과 맞는가, 어느 쪽이 더 시끄러운가(07e).

**합격 기준 — 이 그림·숫자가 나오면 성공.**

1. 유러피언 콜 MC(200,000 경로, FAST 20,000)가 BS 10.450584와 4 SE 이내. 반정 변량이 분산을 약 2.0배, $S_T$ 대조 변량이 약 6.9배 줄이는 것을 SE로 확인(SE vs $n$ log-log 그림에서 기울기 $-1/2$의 평행선 세 개).
2. 기하 아시안 MC가 닫힌 형식 5.641058과 4 SE 이내. 산술 아시안의 대조 변량 추정이 plain보다 SE를 약 35배(분산 약 1,300배) 줄이고 참고값 $5.8576 \pm 0.0002$와 일치.
3. 장벽: $m = 50$ 이산 관측 MC ≈ 9.18(정확 이산값 9.18173, BGK 9.18224)로 연속 공식 8.665472보다 0.52 높다. 브라운 다리 보정 MC는 8.6655와 4 SE 이내. 편향 vs $m$의 log-log 기울기 ≈ $-0.42$(→ $-1/2$).
4. 경로별 Δ = 0.636831, vega = 37.524와 4 SE 이내; 우도비 Γ = 0.018762와 4 SE 이내; 우도비 Δ의 SE가 경로별의 약 2.6배.

**단계.**

| 단계 | 내용 | 도구 |
|---|---|---|
| 1 | 유러피언 콜: plain vs 반정 vs 대조 변량, SE vs $n$ | `mc_option_price`, inline `euro_samples`, `control_variate` |
| 2 | 아시안 콜: 기하 닫힌 형식, 산술의 대조 변량, EM 편향 | `gbm_paths`, `euler_maruyama`, inline `geo_asian_call_closed` |
| 3 | 장벽 콜: 이산화 편향, BGK 보정, 브라운 다리 보정 | inline `down_and_out_call_closed`, `bridge_survival`, `discrete_barrier_price_grid` |
| 4 | 그릭스: 경로별 vs 우도비 vs 유한차분 | inline `greek_estimators` |
| Compare | 모든 검증 수치를 한 표로 | `compare_table`, `assert_close` |
| 5 | 가격표 보고와 방법 추천 | — |
| 스트레치 | S1 아메리칸 풋(LSM vs CRR), S2 중요도 표본화, S3 준몬테카를로 | — |

## 0. 준비 (Setup)

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy.integrate import quad
from scipy.signal import fftconvolve
from IPython.display import Markdown
from spkit import rng_for, fast, scale, setup_plots
from spkit.mc import mc_estimate, compare_table, assert_close
from spkit.sde import black_scholes_call, black_scholes_put, mc_option_price, euler_maruyama
from spkit.brownian import gbm_paths, running_max
from spkit.plots import plot_paths

SEED, rng = rng_for("C4")
setup_plots()
T_START = time.perf_counter()
S0, K, r, sigma, T = 100.0, 100.0, 0.05, 0.2, 1.0     # 블랙–숄즈 세계
H, N_OBS, BETA_BGK = 90.0, 50, 0.5826                  # 장벽, 아시안 관측일 수, BGK 상수
DISC, ES_T = np.exp(-r * T), S0 * np.exp(r * T)        # 할인계수, E*[S_T]
N_PATHS = scale(200_000)                               # FAST: 20,000
N_BARRIER = scale(100_000, minimum=25_000)             # 장벽 MC 경로 수 (FAST: 25,000)
N_GRID = [500, 2_000, 8_000, 20_000] if fast() else [1_000, 4_000, 16_000, 64_000, 200_000]
M_LIST = [10, 25, 50, 100, 250]                        # 장벽 관측 횟수
print(f"SEED={SEED}, fast={fast()}, N_PATHS={N_PATHS}, N_BARRIER={N_BARRIER}")

## 1. Recall — 재사용 노트북에서

책을 덮고 먼저 답을 적어 본 뒤 정답을 펼쳐 보세요.

**Q1.** (07e) 위험중립 가격 공식과 $Q$ 아래 GBM의 SDE를 쓰라. 할인 가격 $e^{-rt}S_t$는 무엇인가?

<details><summary>정답</summary>

$V_0 = E^*[e^{-rT}h(S_T)]$, $dS = rS\,dt + \sigma S\,dW^*$, $S_T = S_0\exp\big((r - \sigma^2/2)T + \sigma W^*_T\big)$. 할인 가격 $e^{-rt}S_t$는 $Q$-마팅게일이다 — 그래서 위험중립 측도 아래에서는 드리프트가 $\mu$가 아니라 $r$이다.

</details>

**Q2.** (07e) 블랙–숄즈 콜 공식과 $d_1, d_2$를 쓰고 $S_0 = K = 100$, $r = 0.05$, $\sigma = 0.2$, $T = 1$의 값을 말하라.

<details><summary>정답</summary>

$c = S_0\Phi(d_1) - Ke^{-rT}\Phi(d_2)$, $d_1 = \big[\ln(S_0/K) + (r + \sigma^2/2)T\big]/(\sigma\sqrt T) = 0.35$, $d_2 = d_1 - \sigma\sqrt T = 0.15$. $c = 10.4506$, 풋은 패리티로 $p = c - S_0 + Ke^{-rT} = 5.5735$.

</details>

**Q3.** (07e) Feynman–Kac: $u(t, x) = E[e^{-r(T-t)}h(S_T) \mid S_t = x]$는 어떤 PDE를 푸는가?

<details><summary>정답</summary>

$u_t + rxu_x + \tfrac12\sigma^2x^2u_{xx} - ru = 0$, 말기 조건 $u(T, x) = h(x)$ — 블랙–숄즈 PDE. MC는 이 PDE를 풀지 않고 확률 표현을 그대로 표본화한다.

</details>

**Q4.** (07d, 두 노트북 전) OU 과정 $dX = \theta(\mu - X)dt + \sigma dW$의 정상분포와 자기공분산은?

<details><summary>정답</summary>

$N(\mu, \sigma^2/(2\theta))$, $\mathrm{Cov}(X_s, X_t) = \frac{\sigma^2}{2\theta}e^{-\theta|t-s|}$. (이 캡스톤에서는 쓰지 않지만, 확률적 변동성 모형에서 $\sigma$ 자체를 OU로 움직이는 것이 다음 단계다.)

</details>

## 2. 이론 — 내 말로 다시 쓰기

### 2.1 정의

- **페이오프.** 유러피언 콜 $(S_T - K)^+$. 이산 관측 아시안 콜 $(A - K)^+$, $A = \frac1n\sum_{i=1}^n S_{t_i}$(산술) 또는 $G = \big(\prod_i S_{t_i}\big)^{1/n}$(기하), $t_i = iT/n$. 다운앤아웃 콜 $(S_T - K)^+\mathbf 1\{\min_{t \le T}S_t > H\}$(연속 관측) 또는 $\mathbf 1\{\min_i S_{t_i} > H\}$(이산 관측, $m$회).
- **MC 추정량.** $\hat V = \frac1n\sum_j e^{-rT}h_j$, SE $= \mathrm{sd}/\sqrt n$ (00b). **정확 GBM 스텝** $S_{t_{i+1}} = S_{t_i}\exp\big((r - \sigma^2/2)\Delta t + \sigma\sqrt{\Delta t}\,Z\big)$ (spkit `gbm_paths`, 편향 없음) vs **오일러–마루야마** $S_{t_{i+1}} = S_{t_i}(1 + r\Delta t + \sigma\sqrt{\Delta t}\,Z)$ (07c, 약수렴 차수 1 → 가격 편향 $O(\Delta t)$).
- **반정 변량.** $Y = h(Z)$, $\tilde Y = h(-Z)$, 추정량 $(Y + \tilde Y)/2$. 분산 $\frac{\mathrm{Var}(Y) + \mathrm{Cov}(Y, \tilde Y)}{2}$. **효율** $= \mathrm{Var}(Y)/(\mathrm{Var}(Y) + \mathrm{Cov})$: 같은 총 경로 수(쌍 하나 = 경로 둘)로 비교했을 때의 분산 감소 배수.
- **대조 변량.** 기댓값 $E[C]$를 아는 $C$로 $Y_b = Y - b(C - E[C])$. $b^* = \mathrm{Cov}(Y, C)/\mathrm{Var}(C)$, $\mathrm{Var}(Y_{b^*}) = \mathrm{Var}(Y)(1 - \rho_{YC}^2)$. $\hat b$를 같은 표본에서 추정하면 $O(1/n)$ 편향(무시 가능).
- **BGK 연속성 보정.** 이산 관측(간격 $\Delta t$) 장벽 $H$의 가격 ≈ 연속 관측 장벽 $He^{-\beta\sigma\sqrt{\Delta t}}$(다운 장벽)의 가격, $\beta = -\zeta(1/2)/\sqrt{2\pi} = 0.5826$. 반대로 이산 MC로 연속 가격을 원하면 장벽을 $He^{+\beta\sigma\sqrt{\Delta t}}$로 **올린다**.
- **브라운 다리 교차확률 (06c).** $X = \ln S$가 $[t_i, t_{i+1}]$ 양 끝에서 $a, b > h = \ln H$일 때 $P(\min X \le h \mid a, b) = \exp\big(-2(a - h)(b - h)/(\sigma^2\Delta t)\big)$ (드리프트 무관). 생존확률 곱 $\prod_i(1 - p_i)$를 페이오프에 곱하면 연속 장벽의 **무편향** 추정량.
- **그릭스.** $\Delta = \partial V/\partial S_0 = \Phi(d_1)$, vega $= \partial V/\partial\sigma = S_0\phi(d_1)\sqrt T$, $\Gamma = \phi(d_1)/(S_0\sigma\sqrt T)$. 경로별: $\hat\Delta = e^{-rT}\mathbf 1\{S_T > K\}S_T/S_0$, $\widehat{\mathrm{vega}} = e^{-rT}\mathbf 1\{S_T > K\}S_T\big(\ln(S_T/S_0) - (r + \sigma^2/2)T\big)/\sigma$. 우도비: $\hat\Delta_{LR} = e^{-rT}h(S_T)\,Z/(S_0\sigma\sqrt T)$, $\hat\Gamma_{LR} = e^{-rT}h(S_T)\Big[\frac{Z^2 - 1}{S_0^2\sigma^2T} - \frac{Z}{S_0^2\sigma\sqrt T}\Big]$, $Z = \big(\ln(S_T/S_0) - (r - \sigma^2/2)T\big)/(\sigma\sqrt T)$.

### 2.2 정리 1 — 반정 변량의 분산 감소 (단조 페이오프) (Glasserman §4.2)

> $h$가 단조이고 $Z \sim N(0, 1)$이면 $\mathrm{Cov}(h(Z), h(-Z)) \le 0$, 따라서 $\mathrm{Var}\big(\tfrac{h(Z) + h(-Z)}2\big) \le \tfrac12\mathrm{Var}(h(Z))$: 반정 쌍 하나가 독립 표본 둘보다 낫다. 콜: 상관 $-0.5010$, 효율 2.004배.

**가정이 왜 필요한가**
- **$h$ 단조**: 비단조 페이오프(스트래들 $|S_T - K|$)에서는 상관이 양수($+0.645$)가 되어 분산이 오히려 1.6배 커진다(E3).
- **$Z$의 분포가 대칭** ($-Z \overset d= Z$): 반정 표본이 같은 분포를 가져야 추정량이 무편향이다.

<details><summary>증명</summary>

$g(z) = h(-z)$는 $h$와 반대 방향으로 단조다. 독립 복제 $Z'$를 잡으면 모든 $z, z'$에 대해 $(h(z) - h(z'))(g(z) - g(z')) \le 0$이다(한쪽이 커질 때 다른 쪽은 작아지므로). 기댓값을 취하면
$$0 \ge E\big[(h(Z) - h(Z'))(g(Z) - g(Z'))\big] = 2E[h(Z)g(Z)] - 2E[h(Z)]E[g(Z)] = 2\,\mathrm{Cov}(h(Z), h(-Z))$$
(교차항 $E[h(Z)g(Z')] = E[h(Z)]E[g(Z')]$는 독립성, $g(Z') \overset d= g(Z)$는 대칭성). 이것이 Chebyshev의 결합 부등식이다. 분산은 $\mathrm{Var}\big(\frac{Y + \tilde Y}2\big) = \frac{\mathrm{Var}(Y) + \mathrm{Var}(\tilde Y) + 2\mathrm{Cov}}4 = \frac{\mathrm{Var}(Y) + \mathrm{Cov}}2 \le \frac{\mathrm{Var}(Y)}2$.

</details>

### 2.3 정리 2 — 대조 변량의 최적 계수 (Glasserman §4.1)

> $Y_b = Y - b(C - E[C])$의 분산은 $\mathrm{Var}(Y) - 2b\,\mathrm{Cov}(Y, C) + b^2\mathrm{Var}(C)$로, $b^* = \mathrm{Cov}(Y, C)/\mathrm{Var}(C)$에서 최소 $\mathrm{Var}(Y)(1 - \rho^2)$가 된다. 유러피언 콜에 $C = S_T$ ($E[S_T] = S_0e^{rT}$): $b^* = 0.64076$, 감소 6.883배. 산술 아시안에 $C$ = 기하 아시안 할인 페이오프: $\rho = 0.99963$, 감소 ≈ 1,300배.

**가정이 왜 필요한가**
- **$E[C]$를 정확히 알아야** 무편향: 기하 아시안의 닫힌 형식(정리 3)이 그 역할을 한다.
- **$\hat b$를 같은 표본으로 추정**하면 $Y$와 $\hat b$가 상관되어 $O(1/n)$ 편향 — 파일럿 표본 또는 큰 $n$으로 무시한다.

<details><summary>증명</summary>

$b$에 대한 2차식 $\mathrm{Var}(Y_b) = \mathrm{Var}(Y) - 2b\,\mathrm{Cov}(Y, C) + b^2\mathrm{Var}(C)$의 도함수를 0으로 놓으면 $b^* = \mathrm{Cov}/\mathrm{Var}(C)$. 대입하면 $\mathrm{Var}(Y) - \mathrm{Cov}^2/\mathrm{Var}(C) = \mathrm{Var}(Y)\big(1 - \rho^2\big)$, $\rho^2 = \mathrm{Cov}^2/(\mathrm{Var}(Y)\mathrm{Var}(C))$. 어떤 $b$를 써도 $E[Y_b] = E[Y]$이므로 무편향은 $E[C]$가 정확한 한 보장된다. 해석: 콜에서 $b^* = 0.641 \approx \Delta = 0.637$ — 대조 변량은 페이오프를 $S_T$에 대해 '선형 회귀'하고 회귀 기울기가 곧 페이오프의 $S_T$에 대한 평균 기울기이기 때문이다.

</details>

### 2.4 정리 3 — 기하 아시안 콜의 닫힌 형식 (이산 관측) (Shreve II §7.5; Glasserman §4.1)

> $\ln G \sim N(m, v^2)$, $m = \ln S_0 + (r - \sigma^2/2)\,T\,\frac{n+1}{2n}$, $v^2 = \sigma^2T\,\frac{(n+1)(2n+1)}{6n^2}$. 가격 $= e^{-rT}\big[e^{m + v^2/2}\Phi(d_1) - K\Phi(d_2)\big]$, $d_2 = (m - \ln K)/v$, $d_1 = d_2 + v$. $n = 50$: 5.641058; $n = 12$: 5.940200; $n \to \infty$: 5.546819.

**가정이 왜 필요한가**
- **GBM의 정확한 로그정규성**: EM 경로에서는 $\ln S$가 정확히 정규가 아니다(편향 $-0.00045$, $n = 50$).
- **등간격 관측일**: 아니면 $m, v^2$가 일반 가중합이 된다.
- **기하 평균**: 산술 평균은 로그정규의 합이라 닫힌 형식이 없다 — 그래서 기하가 대조 변량이 된다.

<details><summary>증명</summary>

$\ln S_{t_i} = \ln S_0 + (r - \sigma^2/2)t_i + \sigma W_{t_i}$이므로 $\ln G = \frac1n\sum_i\ln S_{t_i} = \ln S_0 + (r - \sigma^2/2)\frac1n\sum_i t_i + \frac\sigma n\sum_i W_{t_i}$. 평균: $\frac1n\sum_{i=1}^n t_i = \frac{T}{n^2}\cdot\frac{n(n+1)}2 = T\frac{n+1}{2n}$. 분산: 증분 $\Delta W_j = W_{t_j} - W_{t_{j-1}}$은 $j$번째 이후 모든 항에 등장하므로 $\sum_{i=1}^n W_{t_i} = \sum_{j=1}^n(n - j + 1)\Delta W_j$, 독립 증분의 분산 $T/n$을 곱해
$$\mathrm{Var}\Big(\sum_i W_{t_i}\Big) = \frac Tn\sum_{k=1}^n k^2 = \frac Tn\cdot\frac{n(n+1)(2n+1)}6.$$
$\sigma^2/n^2$을 곱하면 $v^2 = \sigma^2T\frac{(n+1)(2n+1)}{6n^2}$. $n \to \infty$에서 $v^2 \to \sigma^2T/3$, 즉 평균화가 변동성을 $\sigma/\sqrt3$ 수준으로 줄인다. 로그정규 $G$에 대한 $E[e^{-rT}(G - K)^+]$는 BS와 같은 적분: $\ln G = m + vZ$로 쓰면 $E[(e^{m + vZ} - K)^+] = e^{m + v^2/2}\Phi(d_2 + v) - K\Phi(d_2)$, $d_2 = (m - \ln K)/v$(적분 하한 $z > (\ln K - m)/v = -d_2$; 첫 항은 정규 밀도의 평행이동). 노트북에서 $n = 50$의 값을 수치 적분으로 재확인한다.

</details>

### 2.5 정리 4 — 다운앤아웃 콜의 닫힌 형식 (연속 관측, $H \le K$) (Shreve II §7.3)

> $c_{do} = c_{BS} - c_{di}$, $c_{di} = S_0(H/S_0)^{2\lambda}\Phi(y) - Ke^{-rT}(H/S_0)^{2\lambda - 2}\Phi(y - \sigma\sqrt T)$, $\lambda = (r + \sigma^2/2)/\sigma^2$, $y = \ln\frac{H^2}{S_0K}\big/(\sigma\sqrt T) + \lambda\sigma\sqrt T$. $H = 90$: $c_{do} = 8.665472$, $c_{di} = 1.785112$.

**가정이 왜 필요한가**
- **연속 관측**: 이산 관측 가격은 더 높다(녹아웃 기회가 적음): $m = 50$에서 9.18173.
- **$H \le K$이고 $H < S_0$**: 다른 경우는 공식이 다르다(Reiner–Rubinstein 표).

*증명(인용).* 반사 원리(06c)로 $(W_T, \min_{t \le T}W_t)$의 결합 밀도를 구하고 Girsanov로 드리프트를 처리한다; Shreve II §7.3. 본 노트북에서는 브라운 다리 보정 MC(무편향)와 4 SE 이내 일치로 수치 검증한다.

### 2.6 정리 5 — 브라운 다리 교차확률과 BGK 연속성 보정 (Glasserman §6.4; BGK 1997 Thm 1.1)

> (a) 드리프트 있는 BM $X = \ln S$의 두 관측값 $a, b > h$ 사이에서 $h$ 이하로 내려갈 조건부 확률은 $\exp\big(-2(a - h)(b - h)/(\sigma^2\Delta t)\big)$. (b) BGK: $V_{disc}(H) = V_{cont}\big(He^{-\beta\sigma\sqrt{\Delta t}}\big) + o(\sqrt{\Delta t})$, $\beta = 0.5826$. 수치: $m = 50$ 정확 이산 9.18173 vs BGK 9.18224(오차 $5 \times 10^{-4}$); $m = 250$: 8.91481 vs 8.91485(오차 $4 \times 10^{-5}$). BGK 오차는 모든 $m$에서 양수이고 $m$이 커질수록 단조 감소한다.

**가정이 왜 필요한가**
- **(a) 브라운 다리는 드리프트와 무관** — 양 끝을 고정하면 드리프트가 조건부 분포에서 상쇄된다. 그래서 식에 $r$이 없다.
- **(b) $\Delta t \to 0$의 점근 결과**: $m = 10$에서는 오차가 $8.8 \times 10^{-3}$으로 커진다.

<details><summary>증명 스케치</summary>

(a) 반사 원리: 드리프트 없는 BM(분산율 $\sigma^2$)이 $a$에서 출발해 $[0, \Delta t]$ 동안 $h$에 닿고 $b$에서 끝나는 경로는 $h$에서 반사하면 $2h - b$에서 끝나는 경로와 일대일이다. 따라서 조건부 확률은 밀도의 비
$$\frac{\phi\big((2h - b - a)/(\sigma\sqrt{\Delta t})\big)}{\phi\big((b - a)/(\sigma\sqrt{\Delta t})\big)} = \exp\Big(-\frac{(2h - a - b)^2 - (b - a)^2}{2\sigma^2\Delta t}\Big) = \exp\Big(-\frac{2(a - h)(b - h)}{\sigma^2\Delta t}\Big)$$
($(2h - a - b)^2 - (b - a)^2 = 4(a - h)(b - h)$). 드리프트 $\mu$가 있으면 Girsanov 밀도 $\exp(\mu(b - a)/\sigma^2 - \mu^2\Delta t/(2\sigma^2))$가 끝점에만 의존하므로 분자·분모에서 상쇄된다 — 06c의 $P(\max B_t \ge a)$ 계산을 조건부로 반복한 것이다.

(b) 이산 관측은 장벽을 '평균적으로 $\beta\sigma\sqrt{\Delta t}$만큼 덜 본다'는 사실에서 나온다: 랜덤 워크가 장벽을 처음 넘을 때 **초과량**(overshoot)의 기댓값이 $\sigma\sqrt{\Delta t}\cdot\beta$로 수렴한다(Siegmund의 초과 이론; $\beta = -\zeta(1/2)/\sqrt{2\pi}$). 따라서 편향 $V_{disc} - V_{cont}$는 $\sqrt{\Delta t}$ 차수다: $m = 10/25/50/100/250$에서 0.967/0.689/0.516/0.380/0.249, $\times\sqrt m$ = 3.06/3.45/3.65/3.80/3.94(서서히 상수로 수렴).

</details>

### 2.7 정리 6 — 경로별 미분과 우도비 추정량 (Glasserman §7.2–7.3)

> 페이오프가 $S_0$(또는 $\sigma$)에 대해 립시츠이고 a.s. 미분가능하면 $\partial_{S_0}E[h] = E[\partial_{S_0}h]$ (지배수렴) — 콜의 Δ, vega는 OK; Γ와 디지털의 Δ는 불가(불연속 → 경로별 미분 0 a.s.). 우도비 $\partial_\theta E[h] = E[h\,\partial_\theta\ln f_\theta]$는 페이오프 조건 없이 성립하나 분산이 크다(Δ의 SE: LR 0.0033 vs 경로별 0.0013).

**가정이 왜 필요한가**
- **립시츠**: $\partial_{S_0}(S_T - K)^+ = \mathbf 1\{S_T > K\}S_T/S_0$가 유계 도함수를 가지므로 미분과 기대를 교환할 수 있다. 디지털 $\mathbf 1\{S_T > K\}$는 도함수가 0 a.s.이지만 참 Δ는 $e^{-rT}\phi(d_2)/(S_0\sigma\sqrt T) \ne 0$이다.
- **우도비**: 밀도 $f_\theta$가 $\theta$에 매끄럽고 지지집합이 $\theta$에 무관해야 한다(로그정규 OK).

<details><summary>증명 스케치</summary>

경로별: $S_T = S_0e^{(r - \sigma^2/2)T + \sigma\sqrt TZ}$는 $S_0$에 선형이므로 $\partial S_T/\partial S_0 = S_T/S_0$; 체인 룰로 $\partial_{S_0}(S_T - K)^+ = \mathbf 1\{S_T > K\}S_T/S_0$. vega: $\partial S_T/\partial\sigma = S_T(\sqrt TZ - \sigma T) = S_T\big(\ln(S_T/S_0) - (r + \sigma^2/2)T\big)/\sigma$. 우도비: $S_T$의 로그정규 밀도에서 $\ln f = -\frac{Z^2}2 - \ln(S_T\sigma\sqrt{2\pi T})$, $Z = Z(S_T; S_0)$이고 $\partial_{S_0}Z = -1/(S_0\sigma\sqrt T)$이므로 스코어 $\partial_{S_0}\ln f = Z/(S_0\sigma\sqrt T)$. Γ는 스코어를 한 번 더 미분: $\partial_{S_0}^2 f/f = (\partial_{S_0}\ln f)^2 + \partial_{S_0}^2\ln f = \frac{Z^2 - 1}{S_0^2\sigma^2T} - \frac{Z}{S_0^2\sigma\sqrt T}$.

</details>

### 2.8 직관

MC 가격 결정의 오차는 두 종류다. **통계 오차**(SE, $1/\sqrt n$로만 줄어듦)와 **이산화 편향**(경로를 유한 격자로 보는 데서 옴, $n$과 무관). 분산 감소 기법은 첫 번째만 건드리고, 연속성 보정·브라운 다리는 두 번째만 건드린다. '닫힌 형식이 있는 가까운 문제'가 두 곳에서 무기가 된다: 기하 아시안은 산술 아시안의 대조 변량(분산 1,300배 감소), 연속 장벽 공식은 이산 MC의 편향을 재는 자(ruler)다. 그릭스는 '기대의 미분'을 '미분의 기대'로 바꿀 수 있느냐의 문제이고, 못 바꾸면(불연속) 밀도를 대신 미분한다(우도비).

### 2.9 함정(traps)

- 장벽 옵션 MC가 연속 공식보다 높은 것은 **분산이 아니라 편향**이다 — 경로 수를 늘려도 안 줄어든다. $m = 250$(일별)에서도 $+0.25$.
- 반정 변량은 단조 페이오프에서만 이득이다: 스트래들은 분산이 1.6배 커지고, 디지털은 4.7배 줄어든다.
- 대조 변량 계수 $\hat b$를 같은 표본에서 추정한 뒤 그 표본으로 SE를 계산하면 미세 편향($O(1/n)$)이 있다 — $n = 200{,}000$이면 무시 가능하지만 $n$이 작을 때는 파일럿 표본을 써라.
- EM 경로(07c)로 아시안을 가격 결정하면 정확 스텝과 달리 약한 편향($n = 50$: 유러피언 $-0.0025$, 기하 아시안 $-0.0005$)이 생긴다 — GBM은 정확 표본화가 가능하므로 EM을 쓸 이유가 없다; EM은 닫힌 형식이 없는 SDE를 위한 것이다.
- 경로별 Γ는 콜에서 0(꺾인 점 하나가 밀도 0)이 나온다 — 틀린 답. Γ에는 우도비 또는 혼합(pathwise–LR) 추정량.
- 기하 아시안 가격(5.64)이 유러피언(10.45)보다 훨씬 낮은 것은 평균화가 변동성을 $\sigma/\sqrt3$ 수준으로 줄이기 때문 — 버그가 아니다.

✍️ 내 말로: (책을 덮고 여기에 핵심 정의·정리를 다시 써 보세요)

## 3. Predict — 코드를 돌리기 전에 예측

숫자를 적어 보세요. Compare 표의 '내 예측' 열에 들어갑니다(채점 대상은 아닙니다).

In [ ]:
predictions = {
    "bs_call": None,            # BS 콜 가격은? (S0=K=100, r=0.05, sigma=0.2, T=1)
    "bs_put": None,             # BS 풋 가격은? (패리티 c - p = S0 - K e^{-rT} 사용)
    "geo_asian_n50": None,      # 관측일 50개 기하 아시안 콜 가격은? 유러피언보다 얼마나 낮을까?
    "arith_asian_n50": None,    # 산술 아시안 콜은 기하보다 클까 작을까? 값은?
    "antithetic_factor": None,  # 유러피언 콜에서 반정 변량은 분산을 몇 배 줄이는가? (경로 2개 기준)
    "cv_factor_asian": None,    # 산술 아시안에 기하 아시안을 대조 변량으로 쓰면 분산이 몇 배 줄까?
    "barrier_cont": None,       # 다운앤아웃 콜(H=90) 연속 관측 가격은?
    "barrier_disc50": None,     # 50회 이산 관측 가격은? 연속보다 높을까 낮을까?
    "delta": None,              # 콜의 Δ = Φ(d1)은?
    "vega": None,               # 콜의 vega = S0 φ(d1) √T 는?
    "gamma": None,              # 콜의 Γ = φ(d1)/(S0 σ √T) 는?
    "lr_se_ratio": None,        # 우도비 Δ의 SE는 경로별의 몇 배일까?
}

## 4. 단계 1 — 유러피언 콜: plain vs 반정 vs 대조 변량 (07e·00b 재사용)

먼저 닫힌 형식을 손에 쥔다: spkit의 `black_scholes_call`과 직접 계산한 $d_1, d_2$가 같은지, 풋–콜 패리티가 $10^{-10}$까지 맞는지, 그릭스는 얼마인지.

In [ ]:
def bs_d1_d2(S0, K, r, sigma, T):
    d1 = (np.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    return d1, d1 - sigma * np.sqrt(T)

def greeks_closed(S0, K, r, sigma, T):
    d1, d2 = bs_d1_d2(S0, K, r, sigma, T)
    return {"delta": norm.cdf(d1), "vega": S0 * norm.pdf(d1) * np.sqrt(T),
            "gamma": norm.pdf(d1) / (S0 * sigma * np.sqrt(T)),
            "digital": np.exp(-r * T) * norm.cdf(d2)}

d1, d2 = bs_d1_d2(S0, K, r, sigma, T)
C_BS, P_BS = black_scholes_call(S0, K, r, sigma, T), black_scholes_put(S0, K, r, sigma, T)
c_direct = S0 * norm.cdf(d1) - K * DISC * norm.cdf(d2)
np.testing.assert_allclose(C_BS, c_direct, atol=1e-12)
np.testing.assert_allclose(C_BS - P_BS, S0 - K * DISC, atol=1e-10)      # 풋–콜 패리티
GK = greeks_closed(S0, K, r, sigma, T)
print(f"d1 = {d1:.4f}, d2 = {d2:.4f}, call = {C_BS:.6f}, put = {P_BS:.6f}, c - p = {C_BS - P_BS:.6f}")
print(f"delta = {GK['delta']:.6f}, vega = {GK['vega']:.6f}, gamma = {GK['gamma']:.6f}, digital = {GK['digital']:.6f}")

세 추정량. `mc_option_price`(07e)는 결과만 돌려주므로 대조 변량과 그릭스를 위해 표본 $Z, S_T, Y$를 직접 만드는 `euro_samples`를 둔다. 정확한 분산·상관·$b^*$는 정규 밀도에 대한 수치 적분으로 구해 표본 추정과 나란히 놓는다.

In [ ]:
terminal = lambda z, s0=S0, sig=sigma: s0 * np.exp((r - 0.5 * sig**2) * T + sig * np.sqrt(T) * z)
call, put = (lambda s: np.maximum(s - K, 0.0)), (lambda s: np.maximum(K - s, 0.0))

def euro_samples(n, rng):                       # helper (spkit에 없어서 여기서 정의)
    Z = rng.standard_normal(n)
    S_T = terminal(Z)
    return Z, S_T, DISC * call(S_T)

def control_variate(Y, C, EC):                  # helper: Y - b_hat (C - E C), b_hat = Cov/Var
    b_hat = np.cov(Y, C)[0, 1] / np.var(C, ddof=1)
    return mc_estimate(Y - b_hat * (C - EC)), b_hat

def exact_call_moments():                       # helper: 정규 밀도로 수치 적분한 정확값
    y = lambda z: DISC * max(terminal(z) - K, 0.0)
    E = lambda g: quad(lambda z: g(z) * norm.pdf(z), -12, 12, points=[-d2, d2], limit=200)[0]
    EY, EY2, EYYa = E(y), E(lambda z: y(z) ** 2), E(lambda z: y(z) * y(-z))
    ES2, EYS = E(lambda z: terminal(z) ** 2), E(lambda z: y(z) * terminal(z))
    var_y, cov_anti = EY2 - EY**2, EYYa - EY**2
    cov_ys, var_s = EYS - EY * ES_T, ES2 - ES_T**2
    return {"var_y": var_y, "corr_anti": cov_anti / var_y, "eff_anti": var_y / (var_y + cov_anti),
            "b_star": cov_ys / var_s, "ratio_cv": 1.0 / (1.0 - cov_ys**2 / (var_y * var_s))}

EX = exact_call_moments()
mc_plain = mc_option_price(call, S0, r, sigma, T, N_PATHS, rng)
mc_anti = mc_option_price(call, S0, r, sigma, T, N_PATHS, rng, antithetic=True)   # 같은 총 경로 수
mc_put = mc_option_price(put, S0, r, sigma, T, N_PATHS, rng)
Z, S_T, Y = euro_samples(N_PATHS, rng)
mc_cv, b_hat = control_variate(Y, S_T, ES_T)
print(f"plain      {mc_plain}   (BS {C_BS:.6f})\nantithetic {mc_anti}\ncontrol    {mc_cv}\nput        {mc_put}   (BS {P_BS:.6f})")

In [ ]:
Y_anti = DISC * call(terminal(-Z))                # 같은 Z로 만든 반정 짝
ratio_anti, ratio_cv = (mc_plain.se / mc_anti.se) ** 2, (mc_plain.se / mc_cv.se) ** 2
print("| 항목 | 정확값 (수치 적분) | 표본 추정 |\n|---|---|---|")
print(f"| Var(Y) | {EX['var_y']:.3f} | {Y.var(ddof=1):.3f} |")
print(f"| Corr(Y, Y~) 반정 | {EX['corr_anti']:.4f} | {np.corrcoef(Y, Y_anti)[0, 1]:.4f} |")
print(f"| 반정 효율 (분산 감소 배수) | {EX['eff_anti']:.3f} | {ratio_anti:.3f} (SE 비율 제곱) |")
print(f"| b* (C = S_T) | {EX['b_star']:.5f} | {b_hat:.5f} |")
print(f"| 대조 변량 분산 감소 배수 | {EX['ratio_cv']:.3f} | {ratio_cv:.3f} (SE 비율 제곱) |")
assert mc_anti.se < mc_plain.se < 1.5 * mc_anti.se and mc_cv.se < mc_anti.se

같은 세 추정량의 SE를 $n$에 대해 그리면 00b의 그림이 재현되어야 한다: 기울기 $-1/2$의 평행선 세 개, 위아래 간격이 $\sqrt{2.0}$, $\sqrt{6.9}$.

In [ ]:
se = {"plain": [], "antithetic (same total paths)": [], "control variate C = S_T": []}
for n in N_GRID:
    se["plain"].append(mc_option_price(call, S0, r, sigma, T, n, rng).se)
    se["antithetic (same total paths)"].append(mc_option_price(call, S0, r, sigma, T, n, rng, antithetic=True).se)
    Zn, Sn, Yn = euro_samples(n, rng)
    se["control variate C = S_T"].append(control_variate(Yn, Sn, ES_T)[0].se)
n_arr = np.array(N_GRID, float)
fig, ax = plt.subplots()
for name, vals in se.items():
    slope = np.polyfit(np.log(n_arr), np.log(vals), 1)[0]
    ax.loglog(n_arr, vals, "o-", label=f"{name} (fitted slope {slope:.2f})")
ax.loglog(n_arr, np.sqrt(EX["var_y"] / n_arr), "k--", lw=1, label="sd(Y)/sqrt(n), exact")
ax.set_xlabel("number of paths n"); ax.set_ylabel("standard error of price")
ax.set_title("European call: SE vs n for three estimators (slope -1/2)"); ax.legend(); plt.show()

세 선이 평행하다는 것이 핵심이다. 분산 감소는 **상수 배**를 줄일 뿐 $1/\sqrt n$ 법칙 자체는 바꾸지 못한다. 반정 변량은 경로 두 개당 하나의 표본이므로 '같은 총 경로 수' 기준으로 2.0배, 대조 변량은 표본 수를 그대로 둔 채 6.9배다. 대조 변량 계수 $\hat b \approx 0.64 \approx \Delta$인 것도 눈여겨보자 — 페이오프를 $S_T$에 선형 회귀한 기울기다.

## 5. 단계 2 — 아시안 옵션: 기하 닫힌 형식, 산술의 대조 변량, EM 편향 (06e·07c 재사용)

정리 3을 코드로 옮기고 $n = 50$의 값을 로그정규 밀도의 수치 적분으로 재확인한 뒤, `gbm_paths`(위험중립이므로 $\mu = r$)로 50개 관측일의 경로를 만든다.

In [ ]:
def geo_asian_call_closed(S0, K, r, sigma, T, n):   # helper (spkit에 없어서 여기서 정의)
    m = np.log(S0) + (r - 0.5 * sigma**2) * T * (n + 1) / (2 * n)
    v2 = sigma**2 * T * (n + 1) * (2 * n + 1) / (6 * n**2)
    v = np.sqrt(v2)
    dd2 = (m - np.log(K)) / v
    return np.exp(-r * T) * (np.exp(m + 0.5 * v2) * norm.cdf(dd2 + v) - K * norm.cdf(dd2))

GEO_EXACT = geo_asian_call_closed(S0, K, r, sigma, T, N_OBS)
m_g = np.log(S0) + (r - 0.5 * sigma**2) * T * (N_OBS + 1) / (2 * N_OBS)
v_g = sigma * np.sqrt(T * (N_OBS + 1) * (2 * N_OBS + 1) / (6 * N_OBS**2))
geo_quad = DISC * quad(lambda z: max(np.exp(m_g + v_g * z) - K, 0.0) * norm.pdf(z), -12, 12,
                       points=[(np.log(K) - m_g) / v_g], limit=200)[0]
np.testing.assert_allclose(GEO_EXACT, geo_quad, atol=1e-6)
print(f"geometric Asian (n=50): closed form {GEO_EXACT:.6f}, numerical integration {geo_quad:.6f}")
print(f"  n=12: {geo_asian_call_closed(S0, K, r, sigma, T, 12):.6f}, n=250: {geo_asian_call_closed(S0, K, r, sigma, T, 250):.6f}, "
      f"European: {C_BS:.6f} (ratio {GEO_EXACT / C_BS:.2f})")

t_obs, S_paths = gbm_paths(S0, r, sigma, N_PATHS, N_OBS, T, rng)       # mu = r (위험중립)
A = S_paths[:, 1:].mean(axis=1)                                          # 산술 평균 (t0 제외)
G = np.exp(np.log(S_paths[:, 1:]).mean(axis=1))                         # 기하 평균
fig, ax = plt.subplots()
plot_paths(t_obs, S_paths, ax=ax, n_show=6, alpha=0.6)
ax.plot(t_obs[1:], S_paths[0, 1:], "k.", ms=4, label="monitoring dates (path 0)")
ax.axhline(A[0], color="C3", ls="--", label=f"arithmetic mean A = {A[0]:.1f} (path 0)")
ax.axhline(G[0], color="C2", ls=":", label=f"geometric mean G = {G[0]:.1f} (path 0)")
ax.axhline(K, color="gray", lw=0.8, label="strike K")
ax.set_xlabel("t"); ax.set_ylabel("S_t"); ax.set_title("GBM paths with 50 monitoring dates"); ax.legend(fontsize=8); plt.show()

In [ ]:
Y_g, Y_a = DISC * np.maximum(G - K, 0.0), DISC * np.maximum(A - K, 0.0)
mc_geo, mc_arith_plain = mc_estimate(Y_g), mc_estimate(Y_a)
mc_arith_cv, b_asian = control_variate(Y_a, Y_g, GEO_EXACT)
corr_ag = np.corrcoef(Y_a, Y_g)[0, 1]
ratio_cv_asian = (mc_arith_plain.se / mc_arith_cv.se) ** 2
ARITH_REF, ARITH_REF_SE = 5.8576, 0.0002          # 2e6 경로 대조 변량 MC 참고값 (닫힌 형식 없음)
print(f"geometric  MC {mc_geo}   closed form {GEO_EXACT:.6f}")
print(f"arithmetic plain {mc_arith_plain}")
print(f"arithmetic CV    {mc_arith_cv}   reference {ARITH_REF} ± {ARITH_REF_SE}")
print(f"b_hat = {b_asian:.4f}, corr(Y_a, Y_g) = {corr_ag:.5f}, variance ratio = {ratio_cv_asian:.0f}, "
      f"SE ratio = {mc_arith_plain.se / mc_arith_cv.se:.1f}")
assert mc_arith_cv.se < mc_arith_plain.se / 20 and abs(corr_ag - 0.9996) < 0.001

fig, ax = plt.subplots()
for i, (name, e) in enumerate([("arithmetic, plain", mc_arith_plain), ("arithmetic, CV (geometric)", mc_arith_cv), ("geometric, plain", mc_geo)]):
    ax.errorbar(i, e.mean, yerr=1.96 * e.se, fmt="o", capsize=6, label=f"{name}: {e.mean:.4f} ± {e.se:.4f}")
ax.axhline(GEO_EXACT, color="C2", ls="--", label=f"geometric closed form {GEO_EXACT:.4f}")
ax.axhline(ARITH_REF, color="C3", ls=":", label=f"arithmetic reference {ARITH_REF}")
ax.set_xticks([0, 1, 2]); ax.set_xticklabels(["arith plain", "arith CV", "geo plain"])
ax.set_ylabel("price"); ax.set_title(f"Asian call (n = {N_OBS} dates): estimates with 95% CI"); ax.legend(fontsize=8); plt.show()

산술과 기하 페이오프의 상관이 0.9996이라는 것은, 두 평균이 경로마다 거의 같은 방향으로 움직인다는 뜻이다(AM ≥ GM이고 차이는 작다). 정리 2의 $1 - \rho^2 \approx 7.5 \times 10^{-4}$이 곧 1,300배다. 오차 막대 하나로 비교하면 CV 막대는 점 하나로 보인다.

**EM 편향(07c).** 같은 브라운 증분을 `gbm_paths`의 로그 증분에서 복원해 오일러–마루야마 경로를 만들고, 짝지은 차이(paired difference)로 약수렴 편향을 잰다. 정확 스텝은 편향이 없으므로 차이의 평균이 EM의 편향이다.

In [ ]:
dt_obs = T / N_OBS
dW = (np.diff(np.log(S_paths), axis=1) - (r - 0.5 * sigma**2) * dt_obs) / sigma     # 같은 dW 복원
_, S_em = euler_maruyama(lambda t, x: r * x, lambda t, x: sigma * x, S0, T, N_OBS, rng, dW=dW)
Y_em_euro, Y_ex_euro = DISC * call(S_em[:, -1]), DISC * call(S_paths[:, -1])
Y_em_geo = DISC * np.maximum(np.exp(np.log(np.maximum(S_em[:, 1:], 1e-300)).mean(axis=1)) - K, 0.0)
em_bias_euro, em_bias_geo = mc_estimate(Y_em_euro - Y_ex_euro), mc_estimate(Y_em_geo - Y_g)
print(f"EM - exact, European call:     {em_bias_euro}   (reference -0.0025 ± 0.0002, 2e6 paths)")
print(f"EM - exact, geometric Asian:   {em_bias_geo}   (reference -0.0005 ± 0.0001)")
print(f"EM European price alone: {mc_estimate(Y_em_euro)}  vs BS {C_BS:.6f}")

편향 $-0.0025$는 가격 10.45의 $2.4 \times 10^{-4}$ — 작지만 짝지은 차이로는 full 모드에서 3 SE 넘게 검출된다(FAST에서는 SE 수준). 이산화 편향은 경로 수와 무관하게 남는다는 점을 기억하자: 단계 3의 장벽에서는 같은 현상이 0.5라는 훨씬 큰 규모로 나타난다.

## 6. 단계 3 — 장벽 옵션: 이산화 편향, BGK 보정, 브라운 다리 보정 (06c 재사용)

세 가지 닫힌 형식/정확값을 준비한다: 연속 관측 공식(정리 4), BGK 이동 장벽(정리 5b), 그리고 이산 관측 가격의 **정확값** — 로그가격 밀도를 격자에서 $m$번 전파하며 관측일마다 장벽 아래를 0으로 만드는 `discrete_barrier_price_grid`(장벽을 아주 낮추면 BS 가격이 나와야 한다). 격자 세부가 정확도를 좌우한다: 장벽 $h = \ln(H/S_0)$를 **셀 경계**에 놓고 격자점을 $x_j = h + (j + \tfrac12)\Delta x$로 잡으면(셀 중심 격자) 절단 오차가 $O(\Delta x^2)$가 되어 $\Delta x = 2.5 \times 10^{-4}$에서 $10^{-5}$까지 수렴한다. 장벽을 격자점 위에 놓고 그 점을 0으로 만드는 단순한 방법은 $O(\Delta x)$ 오차(여기서는 약 $+4 \times 10^{-4}$)를 남기므로 BGK 오차($5 \times 10^{-4}$ 수준)를 가려 버린다.

In [ ]:
def down_and_out_call_closed(S0, K, H, r, sigma, T):   # helper: Reiner–Rubinstein, H <= K
    lam, sq = (r + 0.5 * sigma**2) / sigma**2, sigma * np.sqrt(T)
    y = np.log(H**2 / (S0 * K)) / sq + lam * sq
    c_di = S0 * (H / S0) ** (2 * lam) * norm.cdf(y) - K * np.exp(-r * T) * (H / S0) ** (2 * lam - 2) * norm.cdf(y - sq)
    return black_scholes_call(S0, K, r, sigma, T) - c_di, c_di

def bgk_barrier(H, sigma, dt, direction=+1):           # helper: H exp(±β σ √dt)
    return H * np.exp(direction * BETA_BGK * sigma * np.sqrt(dt))

def bridge_survival(logS, h, sigma, dt):               # helper: ∏(1 - p_i), 관측값이 h 이하면 0
    a, b = logS[:, :-1], logS[:, 1:]
    p_cross = np.exp(-2.0 * (a - h) * (b - h) / (sigma**2 * dt))
    return np.prod(np.clip(1.0 - p_cross, 0.0, 1.0), axis=1) * (logS.min(axis=1) > h)

def discrete_barrier_price_grid(m, H=H, dx=2.5e-4, x_max=2.5):   # helper: 정확 이산 관측 가격 (셀 중심 격자)
    h, dt = np.log(H / S0), T / m
    n = int(np.ceil((x_max - h) / dx)); x = h + (np.arange(n) + 0.5) * dx   # 장벽 h 가 첫 셀의 아래 경계
    kern = norm.pdf(np.arange(-n + 1, n) * dx, (r - 0.5 * sigma**2) * dt, sigma * np.sqrt(dt)) * dx
    j = int(np.floor(-h / dx - 0.5)); w = -x[j] / dx                          # x = ln(S/S0) = 0 의 단위 질량을
    f = np.zeros(n); f[j] += (1.0 - w) / dx; f[j + 1] += w / dx              # 이웃 두 격자점에 선형 분배
    for _ in range(m):
        f = fftconvolve(f, kern, mode="same")             # h 아래 격자점은 없으므로 자동으로 흡수(0)
    return DISC * np.sum(np.maximum(S0 * np.exp(x) - K, 0.0) * f) * dx

C_DO, C_DI = down_and_out_call_closed(S0, K, H, r, sigma, T)
DISC_EXACT = {m: discrete_barrier_price_grid(m) for m in M_LIST}
BGK_PRED = {m: down_and_out_call_closed(S0, K, bgk_barrier(H, sigma, T / m, -1), r, sigma, T)[0] for m in M_LIST}
np.testing.assert_allclose(discrete_barrier_price_grid(50, H=S0 * np.exp(-2.5)), C_BS, atol=1e-4)   # 장벽 사실상 없음 → BS
np.testing.assert_allclose(discrete_barrier_price_grid(50, dx=5e-4), DISC_EXACT[50], atol=5e-5)               # 격자 수렴 확인
print(f"continuous down-and-out {C_DO:.6f} (down-and-in {C_DI:.6f}, sum = BS {C_DO + C_DI:.6f})")
for m in M_LIST:
    print(f"  m={m:4d}: exact discrete {DISC_EXACT[m]:.5f}  BGK {BGK_PRED[m]:.5f}  (error {BGK_PRED[m] - DISC_EXACT[m]:+.1e})  bias {DISC_EXACT[m] - C_DO:.3f}")

경로 그림 먼저: $m = 50$ 격자에서 장벽에 닿은 경로(빨강)와 살아남은 경로. 격자 사이에서 장벽을 스쳤다 돌아온 경로는 이산 관측이 놓친다 — 그것이 편향의 출처다.

In [ ]:
def barrier_mc(m, n_paths, rng):                        # (naive, BGK-shifted, Brownian-bridge) at m dates
    _, S = gbm_paths(S0, r, sigma, n_paths, m, T, rng)
    pay, run_min = DISC * call(S[:, -1]), -running_max(-S)[:, -1]
    naive = mc_estimate(pay * (run_min > H))
    shifted = mc_estimate(pay * (run_min > bgk_barrier(H, sigma, T / m, +1)))
    bridge = mc_estimate(pay * bridge_survival(np.log(S), np.log(H), sigma, T / m))
    return naive, shifted, bridge, S

mc_bar_naive, mc_bar_bgk, mc_bar_bridge, S_bar = barrier_mc(50, N_BARRIER, rng)
t_bar = np.linspace(0, T, 51)
fig, ax = plt.subplots()
knocked = (-running_max(-S_bar)[:, -1] <= H)
for i in range(12):
    ax.plot(t_bar, S_bar[i], color="C3" if knocked[i] else "C0", alpha=0.7, lw=1)
ax.axhline(H, color="k", ls="--", label=f"barrier H = {H:.0f}"); ax.axhline(K, color="gray", lw=0.8, label="strike K")
ax.set_xlabel("t"); ax.set_ylabel("S_t"); ax.set_title(f"Down-and-out call, m = 50 dates: knocked out (red) vs alive (blue), P(KO) = {knocked.mean():.2f}")
ax.legend(); plt.show()
print(f"naive discrete MC   {mc_bar_naive}   exact discrete {DISC_EXACT[50]:.5f}")
print(f"BGK-shifted barrier {mc_bar_bgk}   (H -> {bgk_barrier(H, sigma, T / 50):.3f}) continuous {C_DO:.6f}")
print(f"Brownian bridge     {mc_bar_bridge}   continuous {C_DO:.6f}")

In [ ]:
bias_mc = {m: barrier_mc(m, N_BARRIER, rng)[0] for m in M_LIST}
m_arr = np.array(M_LIST, float)
bias_exact = np.array([DISC_EXACT[m] - C_DO for m in M_LIST])
slope_bias = np.polyfit(np.log(m_arr), np.log(bias_exact), 1)[0]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.errorbar(m_arr, [bias_mc[m].mean - C_DO for m in M_LIST], yerr=[1.96 * bias_mc[m].se for m in M_LIST],
             fmt="o", capsize=4, label="naive MC minus continuous (95% CI)")
ax1.loglog(m_arr, bias_exact, "k-", label=f"exact discrete minus continuous (slope {slope_bias:.2f})")
ax1.loglog(m_arr, [BGK_PRED[m] - C_DO for m in M_LIST], "C2--", label="BGK prediction")
ax1.loglog(m_arr, bias_exact[0] * np.sqrt(m_arr[0] / m_arr), "gray", ls=":", label="slope -1/2 reference")
ax1.set_xlabel("monitoring dates m"); ax1.set_ylabel("discretization bias"); ax1.set_title("Down-and-out call: bias vs m (log-log)"); ax1.legend(fontsize=8)
names = ["naive\n(discrete)", "BGK-shifted\nbarrier", "Brownian\nbridge"]
for i, e in enumerate([mc_bar_naive, mc_bar_bgk, mc_bar_bridge]):
    ax2.bar(i, e.mean, yerr=1.96 * e.se, capsize=6, color=f"C{i}", alpha=0.8)
ax2.axhline(C_DO, color="k", ls="--", label=f"continuous formula {C_DO:.4f}")
ax2.axhline(DISC_EXACT[50], color="C3", ls=":", label=f"exact discrete (m=50) {DISC_EXACT[50]:.4f}")
ax2.set_xticks(range(3)); ax2.set_xticklabels(names); ax2.set_ylim(8.4, 9.5); ax2.set_ylabel("price")
ax2.set_title("m = 50: three estimators vs closed forms"); ax2.legend(fontsize=8); plt.show()
print("bias * sqrt(m):", np.round(bias_exact * np.sqrt(m_arr), 2), f"  log-log slope {slope_bias:.3f}")

읽는 법. 왼쪽: 편향은 $m$이 10배 늘 때 약 2.6배($10^{0.42}$)만 준다 — $\sqrt{\Delta t}$ 차수이며, $\text{bias} \times \sqrt m$이 3.06 → 3.94로 서서히 상수에 가까워지므로 극한 기울기는 $-1/2$다. BGK 예측선이 정확 이산값 위에 거의 겹친다($m \ge 50$에서 오차 $5 \times 10^{-4}$ 이하, 부호는 항상 $+$이고 $m$에 대해 단조 감소). 오른쪽: 나이브 MC는 정확 **이산** 가격을 맞히고(계약이 이산 관측이면 이것이 정답), 장벽을 $91.5$로 올린 BGK MC와 브라운 다리 MC는 **연속** 가격을 맞힌다. 브라운 다리는 무편향이고, BGK 이동은 $o(\sqrt{\Delta t})$ 근사다.

## 7. 단계 4 — 그릭스: 경로별 vs 우도비 vs 유한차분 (07e 재사용)

단계 1의 표본 $Z, S_T, Y$를 재사용한다. 경로별 Γ를 '시도'해 보면 항등적으로 0이 나온다: $\partial_{S_0}\big[\mathbf 1\{S_T > K\}S_T/S_0\big] = \mathbf 1\{S_T > K\}\big(S_T/S_0^2 - S_T/S_0^2\big) = 0$ — 꺾인 점의 델타함수가 표본에 잡히지 않는다.

In [ ]:
def greek_estimators(Z, S_T, Y):                        # helper (spkit에 없어서 여기서 정의)
    sq, itm = sigma * np.sqrt(T), (S_T > K).astype(float)
    return {"delta_pw": DISC * itm * S_T / S0,
            "vega_pw": DISC * itm * S_T * (np.log(S_T / S0) - (r + 0.5 * sigma**2) * T) / sigma,
            "delta_lr": Y * Z / (S0 * sq),
            "gamma_lr": Y * ((Z**2 - 1.0) / (S0**2 * sigma**2 * T) - Z / (S0**2 * sq)),
            "gamma_pw": DISC * itm * (S_T / S0**2 - S_T / S0**2)}   # 경로별 Γ 시도: 항등적으로 0

greeks_mc = {k: mc_estimate(v) for k, v in greek_estimators(Z, S_T, Y).items()}
for k, e in greeks_mc.items():
    exact = GK[k.split("_")[0]]
    z = f"z = {(e.mean - exact) / e.se:+.2f}" if e.se > 0 else "(SE = 0: identically zero, wrong on purpose)"
    print(f"{k:9s} {e}   exact {exact:.6f}   {z}")
lr_se_ratio = greeks_mc["delta_lr"].se / greeks_mc["delta_pw"].se
print(f"SE ratio LR / pathwise (delta) = {lr_se_ratio:.2f}")

유한차분(finite difference) Δ도 비교한다: 공통 난수(CRN, 같은 $Z$로 $S_0 \pm 1$)와 독립 난수. 콜 페이오프는 $S_0$에 조각별 선형이므로 CRN 중심차분은 경로별 추정량과 거의 같은 값·같은 분산이지만, 독립 난수로는 두 개의 큰 수를 빼는 셈이라 분산이 폭발한다.

In [ ]:
FD_BUMP = 1.0
fd_crn = mc_estimate(DISC * (call(terminal(Z, S0 + FD_BUMP)) - call(terminal(Z, S0 - FD_BUMP))) / (2 * FD_BUMP))
Z_ind = rng.standard_normal(N_PATHS)
fd_ind = mc_estimate(DISC * (call(terminal(Z, S0 + FD_BUMP)) - call(terminal(Z_ind, S0 - FD_BUMP))) / (2 * FD_BUMP))
delta_est = {"pathwise": greeks_mc["delta_pw"], "likelihood ratio": greeks_mc["delta_lr"],
             "finite diff, CRN": fd_crn, "finite diff, independent": fd_ind}
fig, ax = plt.subplots()
for i, (name, e) in enumerate(delta_est.items()):
    ax.errorbar(i, e.mean, yerr=1.96 * e.se, fmt="o", capsize=6, label=f"{name}: {e.mean:.4f} ± {e.se:.4f}")
ax.axhline(GK["delta"], color="k", ls="--", label=f"Phi(d1) = {GK['delta']:.4f}")
ax.set_xticks(range(4)); ax.set_xticklabels(list(delta_est)); ax.set_ylabel("delta")
ax.set_title("Delta estimators with 95% CI"); ax.legend(fontsize=8); plt.show()

경로별과 CRN 유한차분은 사실상 같은 추정량이다(페이오프가 $S_0 \pm 1$ 사이에서 꺾이는 경로만 다르다). 우도비는 페이오프에 아무 조건도 요구하지 않는 대신 무거운 가중치 $Z/(S_0\sigma\sqrt T)$ 때문에 SE가 2.6배 크다 — 표본 수로 환산하면 6.6배다. Γ는 경로별로는 얻을 수 없으니 우도비가 유일한 선택이고, 실제로 참값 0.018762와 4 SE 이내다.

## 8. Compare — 예측 · 시뮬레이션 · 닫힌 형식 비교

In [ ]:
P = predictions
rows = [
    {"name": "European call, plain MC", "predicted": P["bs_call"], "estimate": mc_plain, "exact": C_BS},
    {"name": "European call, antithetic (same total paths)", "predicted": None, "estimate": mc_anti, "exact": C_BS},
    {"name": "European call, control variate C = S_T", "predicted": None, "estimate": mc_cv, "exact": C_BS},
    {"name": "European put, plain MC (parity check)", "predicted": P["bs_put"], "estimate": mc_put, "exact": P_BS},
    {"name": "Geometric Asian call (n = 50)", "predicted": P["geo_asian_n50"], "estimate": mc_geo, "exact": GEO_EXACT},
    {"name": "Arithmetic Asian call (n = 50), CV [ref 5.8576 ± 0.0002]", "predicted": P["arith_asian_n50"], "estimate": mc_arith_cv, "exact": None},
    {"name": "Down-and-out call, naive discrete MC (m = 50) vs exact discrete", "predicted": P["barrier_disc50"], "estimate": mc_bar_naive, "exact": DISC_EXACT[50]},
    {"name": "Down-and-out call, Brownian bridge (m = 50) vs continuous", "predicted": P["barrier_cont"], "estimate": mc_bar_bridge, "exact": C_DO},
    {"name": "Down-and-out call, BGK-shifted barrier MC (m = 50) vs continuous (approx.)", "predicted": None, "estimate": mc_bar_bgk, "exact": C_DO},
    {"name": "Delta, pathwise", "predicted": P["delta"], "estimate": greeks_mc["delta_pw"], "exact": GK["delta"]},
    {"name": "Delta, likelihood ratio", "predicted": None, "estimate": greeks_mc["delta_lr"], "exact": GK["delta"]},
    {"name": "Vega, pathwise", "predicted": P["vega"], "estimate": greeks_mc["vega_pw"], "exact": GK["vega"]},
    {"name": "Gamma, likelihood ratio", "predicted": P["gamma"], "estimate": greeks_mc["gamma_lr"], "exact": GK["gamma"]},
    {"name": "EM (50 steps) minus exact GBM, European, paired [ref -0.0025 ± 0.0002]", "predicted": None, "estimate": em_bias_euro, "exact": None},
]
Markdown(compare_table(rows))

행별로 '불일치'가 뜻하는 것:
- **유러피언 plain/antithetic/CV, 풋**: `mc_option_price`나 `euro_samples`의 $S_T$ 공식($r - \sigma^2/2$)이 틀렸거나 할인이 빠진 것. 셋 중 CV만 어긋나면 $E[S_T] = S_0e^{rT}$를 잘못 넣은 것.
- **기하 아시안**: 정리 3의 $m, v^2$(관측일에 $t_0$을 포함시켰거나 $\frac{n+1}{2n}$을 $\frac12$로 쓴 것)이나 `gbm_paths`의 열 인덱싱 실수.
- **산술 아시안 CV**: 닫힌 형식이 없어 판정은 '—'; 참고값 $5.8576 \pm 0.0002$와 CI가 겹쳐야 한다. 어긋나면 $\hat b$ 계산이나 $E[C]$에 넣은 기하 값이 틀린 것.
- **장벽 나이브 vs 정확 이산**: 격자 전파 헬퍼(커널 중심, 장벽 절단)나 `min` 인덱싱($t_0$ 포함 여부는 $S_0 > H$라 무관). **브라운 다리 vs 연속**: 교차확률의 부호나 $\sigma^2\Delta t$ 누락. **BGK vs 연속**: 근사이므로 1–2 SE 어긋남은 정상.
- **Δ, vega, Γ**: 경로별 추정량의 지시함수·$S_T/S_0$ 인자, 우도비의 스코어 함수 실수. LR Δ만 어긋나면 $Z$를 $S_T$에서 거꾸로 계산하는 식이 틀린 것.
- **EM 짝지은 차이**: 판정 '—'. 참고값과 부호가 다르면 dW 복원식($\Delta\ln S = (r - \sigma^2/2)\Delta t + \sigma\Delta W$)이 틀린 것.

In [ ]:
assert_close(mc_plain, C_BS, k=4, name="European call plain")
assert_close(mc_anti, C_BS, k=4, name="European call antithetic")
assert_close(mc_cv, C_BS, k=4, name="European call control variate")
assert_close(mc_put, P_BS, k=4, name="European put")
assert_close(mc_geo, GEO_EXACT, k=4, name="geometric Asian")
assert_close(mc_arith_cv, ARITH_REF, se=np.hypot(mc_arith_cv.se, ARITH_REF_SE), k=4, name="arithmetic Asian CV vs reference")
assert_close(mc_bar_naive, DISC_EXACT[50], k=4, name="discrete barrier m=50")
assert_close(mc_bar_bridge, C_DO, k=4, name="Brownian-bridge barrier")
assert mc_bar_naive.mean - C_DO > 4 * mc_bar_naive.se, "discretization bias should be statistically visible"
assert_close(greeks_mc["delta_pw"], GK["delta"], k=4, name="delta pathwise")
assert_close(greeks_mc["delta_lr"], GK["delta"], k=4, name="delta LR")
assert_close(greeks_mc["vega_pw"], GK["vega"], k=4, name="vega pathwise")
assert_close(greeks_mc["gamma_lr"], GK["gamma"], k=4, name="gamma LR")
assert lr_se_ratio > 2.0 and abs(greeks_mc["gamma_pw"].mean) < 1e-12
print("all Monte Carlo checks passed (4 SE)")

## 9. 단계 5 — 가격표 보고

숫자는 모두 위에서 계산한 변수에서 가져온다(하드코딩 없음).

In [ ]:
hdr = "| 옵션 | 방법 | 추정 ± SE | 닫힌 형식/참고값 | 분산 감소 배수 | 편향 처리 |\n|---|---|---|---|---|---|\n"
rows5 = [
    ("유러피언 콜", "닫힌 형식; MC(CV)는 검증용", f"{mc_cv.mean:.4f} ± {mc_cv.se:.4f}", f"{C_BS:.6f}",
     f"반정 {ratio_anti:.2f}배, CV {ratio_cv:.1f}배", "없음 ($S_T$ 정확 표본)"),
    ("유러피언 풋", "풋–콜 패리티", f"{mc_put.mean:.4f} ± {mc_put.se:.4f}", f"{P_BS:.6f}", "—", "없음"),
    ("기하 아시안 (n=50)", "닫힌 형식(정리 3); MC로 확인", f"{mc_geo.mean:.4f} ± {mc_geo.se:.4f}", f"{GEO_EXACT:.6f}", "—", "없음 (정확 GBM 표본)"),
    ("산술 아시안 (n=50)", "MC + 기하 대조 변량", f"{mc_arith_cv.mean:.4f} ± {mc_arith_cv.se:.4f}", f"{ARITH_REF} ± {ARITH_REF_SE} (ref)",
     f"{ratio_cv_asian:.0f}배 (ρ = {corr_ag:.4f})", "없음; EM은 피할 것 (편향 ≈ -0.0005)"),
    ("다운앤아웃 콜, 연속 관측", "MC + 브라운 다리 생존확률", f"{mc_bar_bridge.mean:.4f} ± {mc_bar_bridge.se:.4f}", f"{C_DO:.6f}", "—",
     f"무편향; 나이브 MC는 m=50에서 +{DISC_EXACT[50] - C_DO:.3f} 편향"),
    ("다운앤아웃 콜, 이산 관측 m=50", "나이브 MC (계약 자체가 이산 관측)", f"{mc_bar_naive.mean:.4f} ± {mc_bar_naive.se:.4f}", f"{DISC_EXACT[50]:.5f} (격자), BGK {BGK_PRED[50]:.5f}", "—", "불필요"),
    ("Δ, vega", "경로별", f"{greeks_mc['delta_pw'].mean:.4f} ± {greeks_mc['delta_pw'].se:.4f}; {greeks_mc['vega_pw'].mean:.2f} ± {greeks_mc['vega_pw'].se:.2f}",
     f"{GK['delta']:.6f}; {GK['vega']:.4f}", f"우도비 대비 {lr_se_ratio**2:.1f}배", "립시츠 페이오프 → 무편향"),
    ("Γ", "우도비", f"{greeks_mc['gamma_lr'].mean:.5f} ± {greeks_mc['gamma_lr'].se:.5f}", f"{GK['gamma']:.6f}", "—", "경로별은 0 (틀림)"),
]
elapsed = time.perf_counter() - T_START
Markdown(hdr + "\n".join("| " + " | ".join(row) + " |" for row in rows5) +
         f"\n\n총 실행시간 {elapsed:.1f} s (fast={fast()}, N_PATHS={N_PATHS}, N_BARRIER={N_BARRIER}).")

**추천.** 유러피언 → 닫힌 형식(MC는 코드 검증용). 산술 아시안 → 기하 대조 변량(SE 35배 감소, 사실상 공짜). 장벽 → 계약이 연속 관측이면 브라운 다리 보정(무편향), 이산 관측 계약이면 나이브 MC가 곧 정답이고 BGK는 두 세계를 잇는 근사. Δ/vega → 경로별(립시츠), Γ → 우도비(또는 혼합 추정량). EM 스텝은 GBM에서는 쓰지 않는다.

## 10. 연습문제

### E1 계산

손으로: (i) 풋–콜 패리티로 콜 10.4506에서 풋을 구하라. (ii) 만기에 $S_T > K$이면 1을 주는 현금 디지털 콜의 가격 $e^{-rT}\Phi(d_2)$를 계산하라. (iii) 디지털 콜의 경로별 Δ 추정량이 왜 0인지, 참 Δ $= e^{-rT}\phi(d_2)/(S_0\sigma\sqrt T)$는 얼마인지 쓰라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(i) $p = c - S_0 + Ke^{-rT} = 10.4506 - 100 + 95.1229 = 5.5735$ (`black_scholes_put`: 5.573526).

(ii) $e^{-0.05}\Phi(0.15) = 0.951229 \times 0.559618 = 0.532325$.

(iii) $\mathbf 1\{S_T > K\}$는 $S_0$의 계단함수라 도함수가 a.s. 0이다 — $S_T$가 정확히 $K$인 경로는 확률 0이고, 그 밖의 경로에서는 $S_0$을 조금 움직여도 지시함수가 변하지 않는다. 참 Δ $= e^{-rT}\phi(d_2)/(S_0\sigma\sqrt T) = 0.951229 \times 0.394479/20 = 0.018762$. 콜의 Γ와 같은 값인 것은 우연이 아니다: 디지털 $= -\partial_K(\text{콜})$이고, 콜 가격이 $(S_0, K)$에 대해 1차 동차이므로 Euler 관계식 $c = S_0\partial_{S_0}c + K\partial_Kc$를 $S_0$으로 미분하면 $S_0\partial^2_{S_0}c = -K\partial_{S_0}\partial_Kc$, 즉 $S_0 = K$에서 디지털의 Δ $= -\partial_{S_0}\partial_Kc = \partial^2_{S_0}c = \Gamma$다. 우도비 추정량 `DISC * (S_T > K) * Z / (S0 * sigma * np.sqrt(T))`는 이 값을 4 SE 이내로 맞힌다.

```python
digital = DISC * (S_T > K)
print(GK["digital"], mc_estimate(digital))                                  # 0.532325 vs MC
print(np.exp(-r * T) * norm.pdf(d2) / (S0 * sigma * np.sqrt(T)), GK["gamma"])  # 0.018762 둘 다
print(mc_estimate(digital * Z / (S0 * sigma * np.sqrt(T))))                 # LR delta, SE ≈ 0.0001 (full); exact 0.018762
```

</details>

### E2 유도 후 시뮬레이션 검증

정리 3의 $m, v^2$를 $n = 12$(월별 관측)에 대해 직접 유도해 가격을 구하고, `gbm_paths(n_steps=12)`로 MC 검증하라. $n \to \infty$ 극한($v^2 \to \sigma^2T/3$, $m \to \ln S_0 + (r - \sigma^2/2)T/2$)의 가격도 계산하라(Kemna–Vorst).

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

$n = 12$: $m = \ln 100 + 0.03 \cdot \frac{13}{24} = \ln 100 + 0.01625$, $v^2 = 0.04 \cdot \frac{13 \cdot 25}{6 \cdot 144} = 0.0150463$ ($v = 0.12266$) → $d_2 = 0.01625/0.12266 = 0.1325$, 가격 $= e^{-0.05}\big[e^{m + v^2/2}\Phi(d_2 + v) - 100\,\Phi(d_2)\big] = 5.940200$. 극한: $v^2 = 0.04/3 = 0.013333$, $m = \ln 100 + 0.015$ → 5.546819. 관측이 촘촘할수록 가격이 내려가는 이유: 평균의 분산 $v^2$이 $\sigma^2T\frac{(n+1)(2n+1)}{6n^2}$로 단조 감소한다($n = 1$이면 유러피언 $\sigma^2T$). MC(100,000 경로): SE ≈ 0.026이고 4 SE 이내(추정값 자체는 seed에 따라 다르다). 산술 $n = 12$의 참고값은 6.1561.

```python
def geo_limit(S0, K, r, sigma, T):
    m, v2 = np.log(S0) + (r - 0.5 * sigma**2) * T / 2, sigma**2 * T / 3
    v = np.sqrt(v2); dd2 = (m - np.log(K)) / v
    return np.exp(-r * T) * (np.exp(m + v2 / 2) * norm.cdf(dd2 + v) - K * norm.cdf(dd2))
print(geo_asian_call_closed(S0, K, r, sigma, T, 12), geo_limit(S0, K, r, sigma, T))   # 5.940200, 5.546819
_, S12 = gbm_paths(S0, r, sigma, 100_000, 12, T, rng)
G12, A12 = np.exp(np.log(S12[:, 1:]).mean(1)), S12[:, 1:].mean(1)
mc12 = mc_estimate(DISC * np.maximum(G12 - K, 0)); print(mc12)          # SE ≈ 0.026
assert_close(mc12, geo_asian_call_closed(S0, K, r, sigma, T, 12), k=4)
print(mc_estimate(DISC * np.maximum(A12 - K, 0)))                       # SE ≈ 0.027 (ref 6.1561)
```

</details>

### E3 가정을 깨보기

(a) 스트래들 $|S_T - K|$에 반정 변량을 쓰면 분산이 어떻게 되는가? 정리 1의 어느 가정이 깨졌나? (b) 콜의 Γ를 경로별로 추정하면? (c) 대조 변량으로 $S_T$ 대신 상관이 약한 것(예: 딥 OTM 콜 $K = 150$의 페이오프에 $S_T$)을 쓰면 감소가 얼마나 되는가?

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) 정확 계산(수치 적분): $\mathrm{Corr}(h(Z), h(-Z)) = +0.6446$ → 효율 $= 1/(1 + 0.6446) = 0.608$, 즉 같은 경로 수로 분산이 1.64배 **커진다**. 스트래들은 $Z$에 대해 단조가 아니라(V자) 정리 1의 단조성 가정이 깨졌다 — $Z$가 크면 $-Z$도 페이오프가 커서 양의 상관. 반대로 디지털 콜은 단조이고 상관 $-0.787$, 효율 4.69. (b) 0 a.s. — 단계 4에서 보았듯 $\mathbf 1\{S_T > K\}S_T/S_0$의 $S_0$ 도함수는 항등적으로 0이고, 꺾인 점의 2차 미분(델타함수)은 표본에 잡히지 않는다; 우도비 `greeks_mc["gamma_lr"]`는 참값 0.018762와 4 SE 이내다(SE ≈ 0.0003, full 모드). (c) 딥 OTM($K = 150$, BS 0.35963)은 페이오프가 대부분 0이라 $S_T$와의 상관이 약 0.41에 그치고 분산 감소는 $1/(1 - 0.41^2) \approx 1.2$배뿐이다 — 중요도 표본화(S2)가 답이다.

```python
Y_str, Y_str_a = DISC * np.abs(S_T - K), DISC * np.abs(terminal(-Z) - K)
c = np.corrcoef(Y_str, Y_str_a)[0, 1]; print(c, 1 / (1 + c))              # ≈ +0.644, efficiency ≈ 0.61
Y_dig, Y_dig_a = DISC * (S_T > K), DISC * (terminal(-Z) > K)
c = np.corrcoef(Y_dig, Y_dig_a)[0, 1]; print(c, 1 / (1 + c))              # ≈ -0.787, efficiency ≈ 4.69
print(greeks_mc["gamma_pw"], greeks_mc["gamma_lr"])                        # 0 vs LR (exact 0.018762)
Y_otm = DISC * np.maximum(S_T - 150, 0)
e_cv, b_otm = control_variate(Y_otm, S_T, ES_T)
rho = np.corrcoef(Y_otm, S_T)[0, 1]
print(mc_estimate(Y_otm), e_cv, black_scholes_call(S0, 150, r, sigma, T), rho, 1 / (1 - rho**2))   # 0.3596, ratio ≈ 1.2
```

</details>

### 스트레치 S1–S3 (선택)

**S1 (아메리칸 풋, LSM vs CRR).** 아메리칸 풋(같은 파라미터)을 Longstaff–Schwartz 최소제곱 MC(50 조기행사일, 기저 $1, S, S^2$)로 가격 결정하고 CRR 이항트리(2,000–4,000 스텝) 6.090과 비교하라. 유러피언 풋 5.5735와의 차이(조기행사 프리미엄 0.517)를 해석하라.

<details><summary>풀이 개요</summary>

CRR: $u = e^{\sigma\sqrt{\Delta t}}$, $d = 1/u$, $p = (e^{r\Delta t} - d)/(u - d)$, 후진 귀납으로 $V_k = \max\big(e^{-r\Delta t}(pV_{k+1}^{u} + (1-p)V_{k+1}^{d}),\ K - S_k\big)$ → $n = 2000$: 6.08999, $n = 4000$: 6.09018. LSM은 회귀로 근사한 계속가치(continuation value)로 행사 규칙을 정하므로 준최적 행사 → 약간 낮은 편향(6.05–6.09). 프리미엄 0.517은 '기다릴 권리'의 가격이 아니라 '지금 받을 권리'의 가격이다: 풋은 깊이 ITM이면 $K - S$를 지금 받아 이자를 버는 것이 유리하다(콜은 배당이 없으면 조기행사 가치가 0).

```python
def crr_american_put(S0, K, r, sigma, T, n):
    dt = T / n; u = np.exp(sigma * np.sqrt(dt)); d = 1 / u
    p, df = (np.exp(r * dt) - d) / (u - d), np.exp(-r * dt)
    V = np.maximum(K - S0 * u ** (n - np.arange(n + 1)) * d ** np.arange(n + 1), 0.0)
    for k in range(n - 1, -1, -1):
        S_k = S0 * u ** (k - np.arange(k + 1)) * d ** np.arange(k + 1)
        V = np.maximum(df * (p * V[:-1] + (1 - p) * V[1:]), K - S_k)
    return V[0]
print(crr_american_put(S0, K, r, sigma, T, 2000), crr_american_put(S0, K, r, sigma, T, 4000))   # 6.08999, 6.09018
```

</details>

**S2 (중요도 표본화).** 딥 OTM 콜 $K = 150$(BS 0.35963)을 plain MC와 드리프트를 올린 중요도 표본화(측도 변환 $Z \to Z + \theta$, 가중치 $e^{-\theta Z - \theta^2/2}$)로 가격 결정하고 SE를 비교하라. $\theta$는 $E[S_T] \approx K$가 되도록 고르라.

<details><summary>풀이 개요</summary>

$\theta = \big(\ln(K/S_0) - (r - \sigma^2/2)T\big)/(\sigma\sqrt T) = (0.4055 - 0.03)/0.2 = 1.877$. 표본 $Z' = Z + \theta$로 $S_T$를 만들고 페이오프에 우도비 $e^{-\theta Z - \theta^2/2}$를 곱하면 무편향이다. plain은 페이오프가 대부분 0이라 상대 SE가 크고(200k 경로에서 0.359 ± 0.0064), IS는 분산이 약 50배 줄어든다(0.360 ± 0.0009). 검증: `assert_close(mc_is, black_scholes_call(S0, 150, r, sigma, T), k=4)`.

```python
K2 = 150.0; theta = (np.log(K2 / S0) - (r - 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
Y_is = DISC * np.maximum(terminal(Z + theta) - K2, 0) * np.exp(-theta * Z - 0.5 * theta**2)
print(theta, mc_estimate(DISC * np.maximum(S_T - K2, 0)), mc_estimate(Y_is), black_scholes_call(S0, K2, r, sigma, T))
```

</details>

**S3 (준몬테카를로).** `scipy.stats.qmc.Sobol`(scrambled)로 유러피언 콜과 기하 아시안($n = 50$차원)을 준몬테카를로로 가격 결정하고, 무작위 시프트 반복(랜덤화된 QMC) 20회로 오차를 추정해 $N^{-1/2}$(MC) vs 더 빠른 감쇠를 log-log로 비교하라.

<details><summary>풀이 개요</summary>

1차원(유러피언)에서는 QMC 오차 $\sim N^{-1}$(기울기 $-1$ vs MC $-1/2$). 50차원 아시안은 직접 넣으면 이득이 작고, 브라운 다리 구성(06e: $W_T$를 먼저, 중간점을 조건부로)으로 유효 차원을 낮추면 이득이 커진다. `Sobol(d, scramble=True, seed=rng)`로 스크램블 시드를 바꿔 가며 20회 반복해 반복 간 표준편차를 오차로 쓴다(단일 QMC 표본에는 SE가 없다). 정확값과의 비교는 같은 `assert_close`.

</details>

## 11. 정리

1. 위험중립 MC는 정확한 GBM 표본으로 편향 없이 가격을 주고 SE는 $1/\sqrt n$; 유러피언 콜 10.4506, 풋 5.5735(패리티)를 재현했다.
2. 반정 변량은 단조 페이오프에서 분산을 ≈ 2배(콜, 상관 $-0.50$), 대조 변량은 $1 - \rho^2$배로 줄인다: $S_T$로 6.9배, 기하 아시안(닫힌 형식 5.6411)으로 산술 아시안을 1,300배.
3. 장벽 옵션의 이산 관측은 $O(\sqrt{\Delta t})$ 편향을 낳는다($m = 50$: 9.182 vs 연속 8.665); BGK 장벽 이동으로 근사 보정하고 브라운 다리 교차확률로 무편향화한다.
4. 그릭스: 립시츠 페이오프면 경로별(Δ 0.6368, vega 37.52), 아니면 우도비(Γ 0.01876, 디지털 Δ) — 우도비는 분산이 크다(Δ SE 2.6배).
5. MC 오차 = 통계 오차 + 이산화 편향. 분산 감소는 전자만, 보정은 후자만 줄인다 — 어느 쪽인지 먼저 진단하라.

**trap 카드**
- Q: 일별 관측($m = 250$) 장벽 옵션 MC가 연속 공식보다 0.25 높다. 경로 수를 10배로 늘리면 해결되는가?
- A: 아니다. 그것은 분산이 아니라 이산 관측 편향이며 $\sqrt{\Delta t}$ 차수로만 준다(BGK: 장벽을 $e^{-0.5826\sigma\sqrt{\Delta t}}$만큼 옮긴 것과 같다). 경로 수는 SE만 줄인다. 연속 가격이 목표면 브라운 다리 보정을, 계약이 일별 관측이면 그 이산 가격이 정답이다.

**다음 노트북**: C5 — 마지막 캡스톤. CTMC·Gillespie(03a)·분기 과정(01g)·경쟁하는 시계(02a)·MC 오차(00b)를 하나의 확률적 전염병 모형에 결합한다.

log/progress.md 한 줄 템플릿:
`- [ ] C4 캡스톤 4: 옵션 가격 결정 — 날짜: ____ / 예측 적중: __/12 / 단계 1-5 __ / E1 __ E2 __ E3 __ / 메모: ____`